# MicroDuck side-kick dance — Kaggle T4 train

Setup from `KAGGLE.md`, then smoke test, then 1024-env PPO. Checkpoints land under `microduck_rl/logs/` and are copied to `/kaggle/working/artifacts/`.

Logger is TensorBoard (`--agent.logger tensorboard`). wandb needs a login Kaggle does not have.

In [ ]:
import os, sys, subprocess

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
os.environ.setdefault("UV_LINK_MODE", "copy")
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"

gpu = subprocess.check_output(["nvidia-smi", "-L"], text=True)
print(gpu)
if "T4" not in gpu:
    sys.exit("Need Tesla T4 (not P100). Got:\n" + gpu)

!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv python install 3.12

In [ ]:
import os, subprocess

os.chdir("/kaggle/working")
if not os.path.isdir("microduck_rl/.git"):
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/pollen-robotics/microduck_rl.git"])
os.chdir("/kaggle/working/microduck_rl")
subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/pezzonovante7/microduck-sidekick-dance.git", "/tmp/dance"])
subprocess.check_call([
    "cp",
    "/tmp/dance/src/mjlab_microduck/tasks/microduck_sidekick_dance_env_cfg.py",
    "src/mjlab_microduck/tasks/",
])
print("task file:", os.path.isfile("src/mjlab_microduck/tasks/microduck_sidekick_dance_env_cfg.py"))

In [ ]:
init = "src/mjlab_microduck/tasks/__init__.py"
block = """
from .microduck_sidekick_dance_env_cfg import (
    make_microduck_sidekick_dance_env_cfg,
    MicroduckSideKickDanceRlCfg,
)
register_mjlab_task(
    task_id="Mjlab-SideKickDance-Flat-MicroDuck",
    env_cfg=make_microduck_sidekick_dance_env_cfg(),
    play_env_cfg=make_microduck_sidekick_dance_env_cfg(play=True),
    rl_cfg=MicroduckSideKickDanceRlCfg,
    runner_cls=MicroduckOnPolicyRunner,
)
"""
text = open(init).read()
if "Mjlab-SideKickDance-Flat-MicroDuck" not in text:
    open(init, "a").write(block)
    print("registered")
else:
    print("already registered")

In [ ]:
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
%cd /kaggle/working/microduck_rl
!uv sync
!uv run python -c "import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"

In [ ]:
import os, subprocess, sys
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"
%cd /kaggle/working/microduck_rl
cmd = [
    "uv", "run", "train", "Mjlab-SideKickDance-Flat-MicroDuck",
    "--env.scene.num-envs", "64",
    "--agent.max-iterations", "5",
    "--agent.logger", "tensorboard",
]
print("+", " ".join(cmd), flush=True)
r = subprocess.run(cmd, check=False)
if r.returncode != 0:
    sys.exit(f"smoke test failed with code {r.returncode}")
print("smoke ok")

In [ ]:
import os, subprocess, sys
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"
%cd /kaggle/working/microduck_rl
cmd = [
    "uv", "run", "train", "Mjlab-SideKickDance-Flat-MicroDuck",
    "--env.scene.num-envs", "1024",
    "--agent.logger", "tensorboard",
]
print("+", " ".join(cmd), flush=True)
r = subprocess.run(cmd, check=False)
if r.returncode != 0:
    sys.exit(f"train failed with code {r.returncode}")

In [ ]:
import glob, os, shutil
from pathlib import Path

root = Path("/kaggle/working/microduck_rl")
dest = Path("/kaggle/working/artifacts")
dest.mkdir(parents=True, exist_ok=True)
found = []
for p in glob.glob(str(root / "logs" / "**" / "*.pt"), recursive=True):
    found.append(p)
    shutil.copy2(p, dest / os.path.basename(p))
print("checkpoints:", len(found))
for p in sorted(found):
    print(p, os.path.getsize(p))